# Exercise - Pricing Swaptions


# 1. Pricing the Swaption


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## Swaption Vol Data

The file `data/swaption_vol_data_2025-06-30.xlsx` has market data on the implied volatility skews for swaptions. Note that it has several columns:
* `expry`: expiration of the swaption
* `tenor`: tenor of the underlying swap
* `model`: the model by which the volatility is quoted. (All are Black.)
* `-200`, `-100`, etc.: The strike listed as difference from ATM strike (bps). Note that ATM is considered to be the **forward swapa rate** which you can calculate.


Your data: you will use a single row of this data for the `1x4` swaption.
* date: `2025-06-30`
* expiration: 1yr
* tenor: 4yrs


In [3]:
vol = pd.read_excel("../Data/swaption_vol_data_2025-06-30.xlsx")
vol.head()

,reference,instrument,model,date,expiration,tenor,-200,-100,-50,-25,0,25,50,100,200
0,SOFR,swaption,black,2025-06-30,1,1,72.250,46.870,39.100,36.000,33.39,31.300,29.760,28.17,28.660
1,SOFR,swaption,black,2025-06-30,1,2,65.780,44.400,37.970,35.460,33.39,31.750,30.530,29.19,29.300
2,SOFR,swaption,black,2025-06-30,1,3,57.870,40.610,35.560,33.650,32.11,30.920,30.060,29.14,29.290
3,SOFR,swaption,black,2025-06-30,1,4,54.405,38.565,33.925,32.195,30.83,29.805,29.095,28.43,28.885
4,SOFR,swaption,black,2025-06-30,1,5,50.940,36.520,32.290,30.740,29.55,28.690,28.130,27.72,28.480


In [9]:
row = vol[(vol["tenor"] == 4) & (vol["expiration"] == 1)]
row

,reference,instrument,model,date,expiration,tenor,-200,-100,-50,-25,0,25,50,100,200
3,SOFR,swaption,black,2025-06-30,1,4,54.405,38.565,33.925,32.195,30.83,29.805,29.095,28.43,28.885


## Rate Data

The file `data/cap_curves_2025-06-30.xlsx` gives 
* SOFR swap rates, 
* their associated discount factors
* their associated forward interest rates.

You will not need the cap data (flat or forward vols) for this problem.


In [4]:
rate = pd.read_excel("../Data/cap_curves_2025-06-30.xlsx")
rate.head(20)

,tenor,swap rates,spot rates,discounts,forwards,flat vols,fwd vols
0,0.25,0.042353,0.042353,0.989523,NaN,NaN,NaN
1,0.50,0.040859,0.040852,0.979883,0.039351,0.156842,0.156842
2,0.75,0.039391,0.039372,0.971043,0.036414,0.180709,0.201708
3,1.00,0.038115,0.038083,0.962807,0.034217,0.204576,0.240464
4,1.25,0.036704,0.036653,0.955417,0.030938,0.242127,0.328341
5,1.50,0.035655,0.035590,0.948239,0.030280,0.268642,0.336521
6,1.75,0.034942,0.034868,0.941054,0.030542,0.285885,0.336809
7,2.00,0.034453,0.034374,0.933835,0.030919,0.295615,0.328654
8,2.25,0.034000,0.033916,0.926827,0.030248,0.299596,0.312413
9,2.50,0.033750,0.033665,0.919605,0.031414,0.299589,0.296022


## The Swaption

Consider the following swaption with the following features:
* underlying is a fixed-for-floating (SOFR) swap
* the underlying swap has **quarterly** payment frequency
* this is a **payer** swaption, which gives the holder the option to **pay** the fixed swap rate and receive SOFR.


### 1.1
Calculate the (relevant) forward swap rate. That is, the one-year forward 4-year swap rate.




In [5]:
fwd_swap = 4 * (rate.loc[rate["tenor"] == 1, "discounts"].values[0] - rate.loc[rate["tenor"] == 5, "discounts"].values[0]) / rate[(rate["tenor"] >= 1.25) & (rate["tenor"] <= 5)]["discounts"].sum()
fwd_swap

np.float64(0.03269770231881184)

### 1.2
Price the swaptions at the quoted implied volatilites and corresponding strikes, all using the just-calculated forward swap rate as the underlying.

In [12]:
row

,reference,instrument,model,date,expiration,tenor,-200,-100,-50,-25,0,25,50,100,200
3,SOFR,swaption,black,2025-06-30,1,4,54.405,38.565,33.925,32.195,30.83,29.805,29.095,28.43,28.885


In [31]:
from scipy.stats import norm

discount = 0.25 *np.sum(rate[(rate["tenor"] >= 1.25) & (rate["tenor"] <= 5)]["discounts"].values)

row_dict = row.to_dict(orient="records")[0]
strikes = [-200, -100, -50, -25, 0, 25, 50, 100, 200]
expiration = row_dict["expiration"]

for strike in strikes:
    implied_vol = row_dict[strike]/ 100
    K = fwd_swap + strike / 10000
    d_1 = (np.log(fwd_swap / K) + 0.5 * implied_vol ** 2 * (expiration)) / (implied_vol * np.sqrt(expiration))
    d_2 = d_1 - implied_vol * np.sqrt(expiration)
    price = discount * (fwd_swap * norm.cdf(d_1) - K * norm.cdf(d_2)) * 100
    print(f"Strike: {K}, Implied Vol: {implied_vol}, Price: {price}")


Strike: 0.01269770231881184, Implied Vol: 0.54405, Price: 7.271096405156423
Strike: 0.022697702318811838, Implied Vol: 0.38565, Price: 3.9480488326097323
Strike: 0.02769770231881184, Implied Vol: 0.33925, Price: 2.5362346966240406
Strike: 0.03019770231881184, Implied Vol: 0.32195, Price: 1.9431299983348316
Strike: 0.03269770231881184, Implied Vol: 0.30829999999999996, Price: 1.443366149589084
Strike: 0.03519770231881184, Implied Vol: 0.29805, Price: 1.0423913720161138
Strike: 0.03769770231881184, Implied Vol: 0.29095, Price: 0.7365597575253519
Strike: 0.04269770231881184, Implied Vol: 0.2843, Price: 0.35573775278071695
Strike: 0.05269770231881184, Implied Vol: 0.28885, Price: 0.08798291755751579



### 1.3
To consider how the expiration and tenor matter, calculate the prices of a few other swaptions for comparison. 
* No need to get other implied vol quotes--just use the ATM implied vol you have for the swaption above. (Here we are just interested in how Black's formula changes with changes in tenor and expiration.)
* No need to calculate for all the strikes--just do the ATM strike.

Alternate swaptions
* The 3mo x 4yr swaption
* The 2yr x 4yr swaption
* the 1yr x 2yr swaption

Report these values and compare them to the price of the `1y x 4y` swaption.